In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [5]:
# define model
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    total_loss = 0

    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 377.9806
Epoch 2, Loss: 179.5872
Epoch 3, Loss: 128.9567
Epoch 4, Loss: 104.5388
Epoch 5, Loss: 89.7230


In [9]:
# quick evaluation
model.eval()
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        target = target.to(device)
        output = model(data)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()

accuracy = correct / len(test_dataset)
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9736


In [ ]:
def export_to_bin(model, output_dir="weights"):
    os.makedirs(output_dir, exist_ok=True)

    state_dict = model.state_dict()

    for name, param in state_dict.items():
        arr = param.detach().cpu().numpy().astype(np.float32)

        # Save raw binary
        filename = os.path.join(output_dir, name + ".bin")
        arr.tofile(filename)

        print(f"Saved: {filename}, shape={arr.shape}")

def dump_data(
    model,
    dataloader,
    device,
    out_dir="data_reference",
    num_samples=5,
    dump_intermediate=False,
    dump_only_input=False,
    ignore_limit=False
):
    os.makedirs(out_dir, exist_ok=True)

    model.eval()
    count = 0

    stats = {"r1":[None,None],"r2":[None,None], "out":[None,None]}

    with torch.no_grad():
        label_file = os.path.join(out_dir, "labels.txt")
        for data, target in dataloader:
            data = data.to(device)
            data_flat = data.view(data.size(0), -1)

            for i in range(data.size(0)):
                if not ignore_limit and count >= num_samples:
                    return

                x = data_flat[i]

                # --- Always save input ---
                inp = x.cpu().numpy().astype(np.float32)
                inp.tofile(f"{out_dir}/input_{count}.bin")

                # --- Always save label ---
                label = target[i].item()
                # with open(f"{out_dir}/label_{count}.txt", "w") as f:
                #     f.write(str(label))

                # --- Skip everything else if only input requested ---
                if dump_only_input:
                    print(f"Saved input {count}, label={label}")
                    with open(label_file, "a") as f:
                        f.write(f"input_{count}.bin {label}\n")
                    count += 1
                    continue

                # --- Compute forward ---
                l1 = model.fc1(x)
                r1 = F.relu(l1)

                l2 = model.fc2(r1)
                r2 = F.relu(l2)

                out = model.fc3(r2)

                if i == 0:
                    stats["r1"] = [r1.min(),r1.max()]
                    stats["r2"] = [r2.min(),r2.max()]
                    stats["out"] = [out.min(),out.max()]
                else:
                    stats["r1"][0] = min(r1.min(), stats["r1"][0])
                    stats["r2"][0] = min(r2.min(), stats["r2"][0])
                    stats["out"][0] = min(out.min(), stats["out"][0])

                    stats["r1"][1] = max(r1.max(), stats["r1"][1])
                    stats["r2"][1] = max(r2.max(), stats["r2"][1])
                    stats["out"][1] = max(out.max(), stats["out"][1])

                print(f"Stats[{count}]: r1:[{r1.min()}, {r1.max()}] r2:[{r2.min()}, {r2.max()}] out:[{out.min()}, {out.max()}]")


                # --- Save only final output by default ---
                # out.cpu().numpy().astype(np.float32).tofile(
                #     f"{out_dir}/output_{count}.bin"
                # )

                # --- Optionally dump intermediates ---
                if dump_intermediate:
                    l1.cpu().numpy().astype(np.float32).tofile(f"{out_dir}/fc1_linear_{count}.bin")
                    r1.cpu().numpy().astype(np.float32).tofile(f"{out_dir}/fc1_relu_{count}.bin")
                    l2.cpu().numpy().astype(np.float32).tofile(f"{out_dir}/fc2_linear_{count}.bin")
                    r2.cpu().numpy().astype(np.float32).tofile(f"{out_dir}/fc2_relu_{count}.bin")

                # print(f"Saved sample {count}, label={label}")

                count += 1


In [ ]:
# call this after training
export_to_bin(model)

In [7]:
dump_data(model, test_loader, device, out_dir="data_input", dump_only_input=False, ignore_limit = True)

Stats[0]: r1:[0.0, 13.627190589904785] r2:[0.0, 12.744555473327637] out:[-23.63616943359375, 11.306336402893066]
Stats[1]: r1:[0.0, 13.67033576965332] r2:[0.0, 11.867558479309082] out:[-11.165155410766602, 8.154142379760742]
Stats[2]: r1:[0.0, 7.775893211364746] r2:[0.0, 7.474663734436035] out:[-7.607788562774658, 6.3528265953063965]
Stats[3]: r1:[0.0, 15.032809257507324] r2:[0.0, 10.47001838684082] out:[-7.879785537719727, 6.837569236755371]
Stats[4]: r1:[0.0, 13.117414474487305] r2:[0.0, 9.031108856201172] out:[-7.059990882873535, 8.618023872375488]
Stats[5]: r1:[0.0, 9.755873680114746] r2:[0.0, 8.79531478881836] out:[-11.710136413574219, 7.461639881134033]
Stats[6]: r1:[0.0, 12.622385025024414] r2:[0.0, 10.228926658630371] out:[-12.190645217895508, 10.0465726852417]
Stats[7]: r1:[0.0, 12.204767227172852] r2:[0.0, 11.039559364318848] out:[-20.71706199645996, 9.787800788879395]
Stats[8]: r1:[0.0, 17.3292236328125] r2:[0.0, 7.872257709503174] out:[-9.208358764648438, 5.247886657714844]

KeyboardInterrupt: 